# Sync Author Curations

Syncs author curations (add/remove works, set display_name/full_name) from the users Heroku Postgres database to local Databricks Delta tables.

**Sources** (typed views over `openalex_users.public.curations`):
- `author_curations_add_work` — add a work to an author profile
- `author_curations_remove_work` — remove a work from an author profile
- `author_curations_set_display_name` — set a display_name for an author
- `author_curations_set_full_name` — set a full_name (used in matching) for an author

**Targets** (split by consumer):
- `openalex.authors.author_works_curations` — per-author add/remove `work_id` arrays (applied during work-authorship resolution)
- `openalex.authors.author_names_curations` — per-author display_name / full_name (applied during profile assembly / matching)

Modeled on `SyncRasCurations.ipynb` with a guardrail cell added before each MERGE so a silent empty/short source can't mass-delete the Delta table. The `WHEN NOT MATCHED BY SOURCE THEN DELETE` clause is what propagates user-initiated retractions (`DELETE /curations/{id}`), so it stays — the guardrail just refuses to fire it on a suspicious source.

**v0 application status**: works curations have apply logic in Phase 5. Names curations sync to Delta but are not yet applied (`set` action is currently rejected by `POST /curations` validation, so the names views and target will stay empty until producer-side support lands).

## Create target tables (idempotent)

In [ ]:
%sql
-- Per-author rollup of add/remove work curations.
-- Same author_id may appear in BOTH arrays if multiple users disagree;
-- precedence (default: remove wins) is decided at apply time in Phase 5.
CREATE TABLE IF NOT EXISTS openalex.authors.author_works_curations (
    author_id                BIGINT NOT NULL,
    curated_add_work_ids     ARRAY<BIGINT>,
    curated_remove_work_ids  ARRAY<BIGINT>,
    updated_datetime         TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

In [ ]:
%sql
-- Per-author display_name / full_name curations.
-- Latest-wins by source `created` when multiple curators set the same property
-- (aggregation handled in the MERGE source query below).
CREATE TABLE IF NOT EXISTS openalex.authors.author_names_curations (
    author_id             BIGINT NOT NULL,
    curated_display_name  STRING,
    curated_full_name     STRING,
    updated_datetime      TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Sync works curations

In [ ]:
%sql
-- Guardrail: refuse to MERGE if the source has unexpectedly few rows.
-- The empty-source check is conditional on the target already holding rows,
-- so the legitimate startup state (both 0) doesn't fail; only "we had data
-- and now the source is empty" fails.
-- Set guardrails_override=true on the job to bypass the decline check
-- (the empty-when-target-nonempty check is unconditional).
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(DISTINCT author_id) FROM (
    SELECT author_id FROM openalex_users.public.author_curations_add_work
    UNION
    SELECT author_id FROM openalex_users.public.author_curations_remove_work
  )
);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.authors.author_works_curations);

SELECT
  new_count AS new_authors,
  current_count AS current_authors,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 author work-curations but target has ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source author count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
-- Preview what will be synced (per-author rollup of add/remove work_ids)
SELECT
  author_id,
  FILTER(
    ARRAY_AGG(CASE WHEN action_kind = 'add' THEN work_id END),
    x -> x IS NOT NULL
  ) AS curated_add_work_ids,
  FILTER(
    ARRAY_AGG(CASE WHEN action_kind = 'remove' THEN work_id END),
    x -> x IS NOT NULL
  ) AS curated_remove_work_ids,
  COUNT(*) AS num_curations
FROM (
  SELECT author_id, work_id, 'add'    AS action_kind FROM openalex_users.public.author_curations_add_work
  UNION ALL
  SELECT author_id, work_id, 'remove' AS action_kind FROM openalex_users.public.author_curations_remove_work
)
GROUP BY author_id

In [ ]:
%sql
-- MERGE works curations (handles inserts, updates, AND deletes via NOT MATCHED BY SOURCE)
MERGE INTO openalex.authors.author_works_curations AS target
USING (
  SELECT
    author_id,
    FILTER(
      ARRAY_AGG(CASE WHEN action_kind = 'add' THEN work_id END),
      x -> x IS NOT NULL
    ) AS curated_add_work_ids,
    FILTER(
      ARRAY_AGG(CASE WHEN action_kind = 'remove' THEN work_id END),
      x -> x IS NOT NULL
    ) AS curated_remove_work_ids,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM (
    SELECT author_id, work_id, 'add'    AS action_kind FROM openalex_users.public.author_curations_add_work
    UNION ALL
    SELECT author_id, work_id, 'remove' AS action_kind FROM openalex_users.public.author_curations_remove_work
  )
  GROUP BY author_id
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Sync names curations

In [ ]:
%sql
-- Guardrail: same shape as the works guardrail. The source for names is the
-- UNION of distinct author_ids across the two `set_*name` views.
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(DISTINCT author_id) FROM (
    SELECT author_id FROM openalex_users.public.author_curations_set_display_name
    UNION
    SELECT author_id FROM openalex_users.public.author_curations_set_full_name
  )
);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.authors.author_names_curations);

SELECT
  new_count AS new_authors,
  current_count AS current_authors,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 author name-curations but target has ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source author count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
-- Preview what will be synced (latest-wins per author for each name property)
WITH latest_display_name AS (
  SELECT author_id, new_display_name AS curated_display_name
  FROM (
    SELECT
      author_id,
      new_display_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_curations_set_display_name
  )
  WHERE rn = 1
),
latest_full_name AS (
  SELECT author_id, new_full_name AS curated_full_name
  FROM (
    SELECT
      author_id,
      new_full_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_curations_set_full_name
  )
  WHERE rn = 1
)
SELECT
  COALESCE(d.author_id, f.author_id) AS author_id,
  d.curated_display_name,
  f.curated_full_name
FROM latest_display_name d
FULL OUTER JOIN latest_full_name f USING (author_id)

In [ ]:
%sql
-- MERGE names curations (handles inserts, updates, AND deletes via NOT MATCHED BY SOURCE)
MERGE INTO openalex.authors.author_names_curations AS target
USING (
  WITH latest_display_name AS (
    SELECT author_id, new_display_name AS curated_display_name
    FROM (
      SELECT
        author_id,
        new_display_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_curations_set_display_name
    )
    WHERE rn = 1
  ),
  latest_full_name AS (
    SELECT author_id, new_full_name AS curated_full_name
    FROM (
      SELECT
        author_id,
        new_full_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_curations_set_full_name
    )
    WHERE rn = 1
  )
  SELECT
    COALESCE(d.author_id, f.author_id) AS author_id,
    d.curated_display_name,
    f.curated_full_name,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM latest_display_name d
  FULL OUTER JOIN latest_full_name f USING (author_id)
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
-- Works target: row counts and array sizes
SELECT
  COUNT(*)                         AS total_curated_authors,
  SUM(SIZE(curated_add_work_ids))    AS total_adds,
  SUM(SIZE(curated_remove_work_ids)) AS total_removes,
  MAX(updated_datetime)            AS last_sync
FROM openalex.authors.author_works_curations

In [ ]:
%sql
-- Sample of curated authors (works)
SELECT * FROM openalex.authors.author_works_curations
ORDER BY updated_datetime DESC
LIMIT 100

In [ ]:
%sql
-- Names target: row counts
SELECT
  COUNT(*)                                                AS total_curated_authors,
  COUNT(curated_display_name)                             AS total_display_names,
  COUNT(curated_full_name)                                AS total_full_names,
  MAX(updated_datetime)                                   AS last_sync
FROM openalex.authors.author_names_curations

In [ ]:
%sql
-- Sample of curated authors (names)
SELECT * FROM openalex.authors.author_names_curations
ORDER BY updated_datetime DESC
LIMIT 100